# Assistente de Voz Multi-Idiomas Com Whisper e Nemotron 3 Super (OpenRouter)

In [52]:
# @title
# Configuração de Tradução Simultânea
# source_language: idioma em que o usuário fala (entrada de áudio)
# response_language: idioma em que o assistente deve responder
# Exemplos: 'pt' (português), 'es' (espanhol), 'en' (inglês), 'fr' (francês)
source_language = 'pt'    # O usuário fala em espanhol
response_language = 'en'  # O assistente responde em português

# Mapeamento de códigos de idioma para nomes completos (usado no prompt do ChatGPT)
LANGUAGE_NAMES = {
    'pt': 'português',
    'es': 'espanhol',
    'en': 'inglês',
    'fr': 'francês',
    'de': 'alemão',
    'it': 'italiano',
    'ja': 'japonês',
    'zh': 'chinês',
}

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [53]:
# @title
# Referência: https://gist.github.com/korakot/c21c3476c024ad6d56d5f48b0bca92be

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

# Código JavaScript para gravar áudio do usuário usando a "MediaStream Recording API"
RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  # Executa o código JavaScript para gravar o áudio
  display(Javascript(RECORD))
  # Recebe o áudio gravado como resultado do JavaScript
  js_result = output.eval_js('record(%s)' % (sec * 1000))
   # Decodifica o áudio em base64
  audio = b64decode(js_result.split(',')[1])
  # Salva o áudio em um arquivo
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  # Retorna o caminho do arquivo de áudio (pasta padrão do Google Colab)
  return f'/content/{file_name}'

# Grava o áudio do usuário por um tempo determinado (padrão 5 segundos)
print('Ouvindo...\n')
record_file = record()

# Exibe o áudio gravado
display(Audio(record_file, autoplay=False))

Ouvindo...



<IPython.core.display.Javascript object>

# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [54]:
# @title
# !pip install git+https://github.com/openai/whisper.git -q

In [55]:
# @title
import whisper

# Selecione o modelo do Whisper que melhor atenda às suas necessidades:
# https://github.com/openai/whisper#available-models-and-languages
model = whisper.load_model("small")

# Transcreve o audio gravado anteriormente.
# Transcreve usando o idioma de origem (source_language)
result = model.transcribe(record_file, fp16=False, language=source_language)
transcription = result["text"]
print(transcription)

 Quem vai ganhar a Copa do Mundo? 2026


# 3. Integração com a API do Nemotron 3 Super via OpenRouter 💬

In [56]:
# @title
# !pip install openai -q

In [57]:
# @title
import os
from openai import OpenAI

# Documentação OpenRouter: https://openrouter.ai/docs/quickstart
# Modelo: Nvidia Nemotron 3 Super 120B (free) - https://openrouter.ai/nvidia/nemotron-3-super-120b-a12b:free

# Para gerar uma API Key do OpenRouter:
# 1. Crie uma conta em https://openrouter.ai
# 2. Acesse https://openrouter.ai/settings/keys
# 3. Clique em "Create Key"

# Substitua o texto "TODO" por sua API Key do OpenRouter.
os.environ['OPENROUTER_API_KEY'] = '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'

# Configura o cliente OpenAI apontando para a API do OpenRouter
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get('OPENROUTER_API_KEY'),
)

# Monta o prompt de sistema para tradução simultânea
source_name = LANGUAGE_NAMES.get(source_language, source_language)
response_name = LANGUAGE_NAMES.get(response_language, response_language)

system_prompt = (
    f"Você é um assistente de tradução simultânea. "
    f"O usuário vai falar em {source_name}. "
    f"Você deve: 1) Traduzir o que o usuário disse para {response_name}. "
    f"2) Responder à pergunta ou mensagem do usuário em {response_name}. "
    f"Formato da resposta:\n"
    f"**Tradução:** [tradução do que o usuário disse]\n"
    f"**Resposta:** [sua resposta ao conteúdo da mensagem]"
)

print(f'\n🌐 Tradução Simultânea: {source_name} → {response_name}\n')

# Envia uma requisição à API do OpenRouter usando o modelo Nemotron 3 Super (free)
# com o prompt de sistema configurado para tradução simultânea.
response = client.chat.completions.create(
    model="nvidia/nemotron-3-super-120b-a12b:free",
    messages=[
        { "role": "system", "content": system_prompt },
        { "role": "user", "content": transcription }
    ]
)

# Obtém a resposta gerada pelo Nemotron 3 Super
nemotron_response = response.choices[0].message.content
print(nemotron_response)


🌐 Tradução Simultânea: português → inglês

**Tradução:** Who will win the World Cup? 2026  
**Resposta:** As of now, it’s impossible to know for certain which team will win the 2026 FIFA World Cup, since the tournament is still several years away and many factors—such as player development, injuries, coaching changes, and qualification outcomes—can shift the balance of power. However, based on current form, talent pools, and recent performances, a few national teams are often mentioned as early favorites:

- **Argentina** – Reigning champions (2022) with a core of world‑class players like Lionel Messi (though he may be older by 2026) and emerging talents such as Julián Álvarez and Enzo Fernández.  
- **Brazil** – Historically strong, consistently producing top‑tier attackers and a deep squad; they will likely be contenders if they maintain their recent defensive solidity.  
- **France** – Winners in 2018 and runners‑up in 2022, featuring a blend of experienced stars (Kylian Mbappé, An

# 4. Sintetizando a Resposta do Nemotron 3 Super Como Voz (gTTS) 🔊


In [58]:
# @title
# !pip install gTTS

In [59]:
# @title
from gtts import  gTTS

# Cria um objeto gTTS com a resposta gerada pelo Nemotron 3 Super no idioma de resposta (response_language).
gtts_object = gTTS(text=nemotron_response, lang=response_language, slow=False)

# Salva o áudio da resposta no arquivo especificado (pasta padrão do Google Colab)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)

# Reproduz o áudio da resposta salvo no arquivo
display(Audio(response_audio, autoplay=True))